Utilizzando il codice visto nella lezione, esegui le seguenti modifiche per comprendere i trade-off profondi:

Analisi del Gradiente: In PyTorch, dopo la chiamata loss.backward(), stampa il valore del gradiente del layer lineare (pt_model.linear.weight.grad) in Keras, prova a fare lo stesso utilizzando un tf.GradientTape (o la versione Keras3 keras.GradintTape) per vedere come il controllo manuale cambi tra i due framework.

Salvataggio e Caricamento: prova a caricare il modello Keras utilizzando keras.models.load_model. Verifica se il modello caricato mantiene le stesse performance del modello originale su un piccolo set di test

Cambio Backend: modifica la variabile d'ambiente KERAS_BACKEND in 'tensorflow' (se installato) o jax. Osserva se il codice Keras continua a funzionare senza modifiche strutturali.

In [ ]:
import os

# 1. CAMBIO BACKEND: Cambia in "jax" o "tensorflow" per testare l'agnosticismo
os.environ["KERAS_BACKEND"] = "torch" 

import keras
import torch
import torch.nn as nn
import numpy as np

# 1. GENERAZIONE DATI SINTETICI (y = 3x + 2)
X = np.random.rand(1000, 1).astype("float32")
y = 3 * X + 2 + np.random.randn(1000, 1).astype("float32") * 0.1

# --- APPROCCIO A: KERAS 3 (Integrazione Nativa con PyTorch) ---
print("\n--- Analisi Keras 3 (Backend: Torch) ---")

keras_model = keras.Sequential([
    keras.layers.Input(shape=(1,)),
    keras.layers.Dense(1)
])

# Convertiamo i dati in tensori del backend corrente (Torch)
X_k = keras.ops.convert_to_tensor(X)
y_k = keras.ops.convert_to_tensor(y)

# CALCOLO GRADIENTI SENZA TAPE:
# Usiamo direttamente l'autograd di PyTorch sul modello Keras!
preds = keras_model(X_k)
loss_k = keras.ops.mean(keras.ops.square(y_k - preds))

# Chiamata nativa PyTorch su output Keras
loss_k.backward() 

# Accediamo ai gradienti dei pesi (trainable_weights[0] è il kernel Dense)
# .value è il tensore Torch sottostante, .grad è il suo gradiente
keras_grad = keras_model.trainable_weights[0].value.grad
print(f"Gradienti Keras (Accesso Nativo): {keras_grad.item():.4f}")

# Training Rapido
keras_model.compile(optimizer="adamw", loss="mse")
keras_model.fit(X, y, epochs=5, batch_size=32, verbose=0)

# SALVATAGGIO E CARICAMENTO
keras_model.save("modello_epicode_2026.keras")
loaded_model = keras.models.load_model("modello_epicode_2026.keras")

# Verifica Performance
perf = loaded_model.evaluate(X, y, verbose=0)
print(f"Modello caricato con successo. Loss su test: {perf:.4f}")


# --- APPROCCIO B: PYTORCH NATIVO ---
print("\n--- Analisi PyTorch Nativo ---")
X_pt = torch.from_numpy(X)
y_pt = torch.from_numpy(y)

class LinearResearchModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
    def forward(self, x):
        return self.linear(x)

pt_model = LinearResearchModel()
optimizer = torch.optim.AdamW(pt_model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Training Loop Manuale
for epoch in range(5):
    optimizer.zero_grad()
    pred = pt_model(X_pt)
    loss = criterion(pred, y_pt)
    loss.backward()
    
    if epoch == 0:
        print(f"Gradienti PyTorch (Layer Lineare): {pt_model.linear.weight.grad.item():.4f}")
    
    optimizer.step()

print(f"Ultima Loss PyTorch: {loss.item():.4f}")